In [50]:
import numpy as np
import copy
import itertools
from itertools import permutations
from collections import Counter

In [51]:
def create_symmetric_array(n):
    if n < 1:
        return []
    first_half = list(range(n, 0, -2))
    second_half = first_half[::-1]
    if len(first_half) + len(second_half) > n:
        second_half = second_half[1:]
    return first_half + second_half


def calculate_happiness(preferences, final_ranking):
    final_ranking_array = list(final_ranking.keys())
    n = len(preferences[0])  # Number of candidates
    happiness_scores = []
    array = create_symmetric_array(n)  # Get the weight array
    max_score = sum(x * n for x in array)  # Max score
    min_score = sum(array[i] * 1 for i in range(n))  # Min score (candidate is last)

    for voter in preferences:
        try:
            if voter[0] == final_ranking_array[0]:
                pos_score = max_score
            else:
                # Compute Positional Satisfaction Score
                pos_score = sum(
                    array[i] * (n - abs(voter.index(c) - final_ranking_array.index(c)))
                    for i, c in enumerate(voter)
                )
        except ValueError:
            # If a candidate is not found, simulate them as being last
            pos_score = min_score

        happiness = round(pos_score / max_score, 2)  # Normalization step
        happiness_scores.append(happiness)

    return happiness_scores


def calculate_total_happiness(hapiness_list):
    """Display the third output."""
    return np.sum(hapiness_list)


def calculate_voting_outcome(voting_scheme, preferences):
    """Display the first output."""
    # clear_screen(root)
    # create an outcome dictionary with all letter (as much as num_candidates) and set their values to 0
    outcome = {}
    num_candidates = len(preferences[0])
    for i in range(num_candidates):
        outcome[chr(65 + i)] = 0

    if voting_scheme == "Plurality":
        for preference in preferences:
            # give 1 point to each candidate in first position of each preference list
            if preference[0] in outcome:
                outcome[preference[0]] += 1
            else:
                outcome[preference[0]] = 1
    elif voting_scheme == "Vote For 2":
        for preference in preferences:
            # give 1 point to each candidate in first 2 positions of each preference list
            if preference[0] in outcome:
                outcome[preference[0]] += 1
            else:
                outcome[preference[0]] = 1
            if preference[1] in outcome:
                outcome[preference[1]] += 1
            else:
                outcome[preference[1]] = 1

    elif voting_scheme == "Anti-Plurality":
        for preference in preferences:
            # give 1 point to each candidate except the last position of each preference list
            for candidate in preference[:-1]:
                if candidate in outcome:
                    outcome[candidate] += 1
                else:
                    outcome[candidate] = 1

    elif voting_scheme == "Borda":
        for preference in preferences:
            # give points based on position in each preference list
            for i, candidate in enumerate(preference):
                if candidate in outcome:
                    outcome[candidate] += num_candidates - i - 1
                else:
                    outcome[candidate] = num_candidates - i - 1
    # order the outcome by number of votes
    outcome = dict(sorted(outcome.items(), key=lambda x: x[1], reverse=True))
    return outcome


def get_strategic_voting_options(
    voting_scheme, outcome, preferences, hapiness_list, num_voters, num_candidates
):
    """Return a structured list of strategic voting options for each voter that increases their happiness level."""
    strategies = ["compromising", "burying", "bullet"]
    strategic_options = []

    for i in range(num_voters):
        voter_strategic_options = []
        for strategy in strategies:
            for candidate in preferences[i]:
                new_preferences = preferences[:]
                new_preferences[i] = strategic_vote(
                    preferences[i],
                    strategy,
                    favored=candidate,
                    disfavored=preferences[i][-1],
                )

                if new_preferences[i] == None:
                    continue

                new_outcome = calculate_voting_outcome(voting_scheme, new_preferences)

                if outcome == new_outcome:
                    continue

                new_hapiness_list = calculate_happiness(preferences, new_outcome)
                if new_hapiness_list[i] > hapiness_list[i]:
                    voter_strategic_options.append(
                        (
                            strategy,
                            new_preferences[i],
                            new_outcome,
                            new_hapiness_list[i],
                            hapiness_list[i],
                            float(np.sum(new_hapiness_list)),
                            float(np.sum(hapiness_list)),
                        )
                    )
        if len(voter_strategic_options) != 0:
            strategic_options.append(
                {"Voter": i + 1, "Strategic Options": voter_strategic_options}
            )
    return strategic_options


def strategic_vote(preference, strategy, favored=None, disfavored=None):
    """
    Modify a voter's preference strategically.

    :param preference: List of ranked candidates (e.g., ['C', 'E', 'A', 'D', 'B'])
    :param strategy: One of 'compromising', 'burying', or 'bullet'
    :param favored: Candidate to favor in 'compromising' or 'burying'
    :param disfavored: Candidate to demote in 'burying'
    :return: Modified preference list
    """
    new_preference = preference[:]

    if strategy == "compromising" and favored:
        # Move favored candidate higher in the ranking
        if favored in new_preference and new_preference.index(favored) > 0:
            new_preference.remove(favored)
            new_preference.insert(0, favored)

            return new_preference

    elif strategy == "burying" and favored and disfavored:
        # Move disfavored candidate lower in the ranking
        if favored in new_preference and disfavored in new_preference:
            new_preference.remove(disfavored)
            new_preference.append(disfavored)

            return new_preference

    elif strategy == "bullet":
        # Only vote for the top choice
        new_preference = [new_preference[0]]
        return new_preference

    return None


def classify_strategic_vote(honest_vote, strategic_vote):

    # Identify the top choice in both votes
    honest_top = honest_vote[0]
    strategic_top = strategic_vote[0]

    # Check for Bullet Voting (if only the top choice is unchanged & others are rearranged randomly)
    if strategic_top == honest_top and sorted(strategic_vote[1:]) == sorted(
        honest_vote[1:]
    ):
        return "Bullet Voting"

    # Identify which candidates moved up or down in ranking
    ranking_changes = {
        candidate: strategic_vote.index(candidate) - honest_vote.index(candidate)
        for candidate in honest_vote
    }

    # Check for Compromising (Top choice is moved down to boost another candidate)
    if honest_top != strategic_top:
        return "Compromising"

    # Check for Burying (A competitor is moved significantly lower)
    for candidate, change in ranking_changes.items():
        if change < 0:  # Candidate moved up in ranking
            for weaker_candidate in honest_vote[honest_vote.index(candidate) + 1 :]:
                if (
                    ranking_changes[weaker_candidate] > 0
                ):  # A lower-ranked candidate was pushed down
                    return "Burying"

    return "Unknown"

In [52]:
def weighted_sampling(num_voters, candidates, preferences):
    """
    Generates plausible full rankings for voters based on observed first-choice votes.
    """
    sampled_rankings = []

    for voter_prefs in preferences:
        first_choice = voter_prefs[0]  # We know each voter's first choice

        # Generate a weighted random sample of the remaining candidates without replacement
        remaining_candidates = [c for c in candidates if c != first_choice]
        weights = np.array(
            [1 / (i + 1) for i in range(len(remaining_candidates))]
        )  # Example weighting: prefer higher-ranked candidates
        weights /= weights.sum()  # Normalize weights
        weighted_remaining = list(
            np.random.choice(
                remaining_candidates,
                size=len(remaining_candidates),
                replace=False,
                p=weights,
            )
        )

        full_ranking = [first_choice] + weighted_remaining
        sampled_rankings.append(full_ranking)

    return sampled_rankings


def atva3(
    voting_scheme, outcome, preferences, happiness_list, num_voters, num_candidates
):
    """
    Estimates full rankings from observed first-choice votes and determines strategic voting options.
    """
    # Extract first-choice votes, sorting alphabetically in case of ties
    first_choice_counts = dict(sorted(Counter(pref[0] for pref in preferences).items()))
    print("First-choice counts:", first_choice_counts)

    # Extract unique candidates
    candidates = sorted(set(c for pref in preferences for c in pref))

    # Generate plausible full rankings using weighted sampling
    estimated_preferences = weighted_sampling(num_voters, candidates, preferences)
    print("Sampled full preferences:", estimated_preferences)

    # Calculate happiness based on preferences
    happiness_list = calculate_happiness(preferences, outcome)
    print("Happiness list:", happiness_list)

    total_happiness = calculate_total_happiness(happiness_list)
    print("Total happiness:", total_happiness)

    # Determine strategic voting options
    strategic_vote = get_strategic_voting_options(
        voting_scheme,
        outcome,
        estimated_preferences,
        happiness_list,
        num_voters,
        num_candidates,
    )
    print("Strategic voting options:", strategic_vote)

    return strategic_vote


preference = [
    ["C", "E", "A", "D", "B"],
    ["B", "A", "D", "E", "C"],
    ["D", "B", "C", "E", "A"],
    ["D", "A", "B", "E", "C"],
]
outcome = calculate_voting_outcome("Borda", preference)
print("Outcome:", outcome)
atva3(
    "Borda",
    outcome,
    preference,
    [1, 1, 1, 1, 1],
    len(preference),
    len(preference[0]),
)

Outcome: {'D': 11, 'B': 9, 'A': 8, 'C': 6, 'E': 6}
First-choice counts: {'B': 1, 'C': 1, 'D': 2}
Sampled full preferences: [['C', 'A', 'E', 'B', 'D'], ['B', 'C', 'A', 'D', 'E'], ['D', 'B', 'C', 'A', 'E'], ['D', 'A', 'B', 'C', 'E']]
Happiness list: [0.44, 0.79, 1.0, 1.0]
Total happiness: 3.23
Strategic voting options: [{'Voter': 1, 'Strategic Options': [('compromising', ['A', 'C', 'E', 'B', 'D'], {'A': 10, 'B': 10, 'C': 9, 'D': 9, 'E': 2}, 0.69, 0.44, 3.0300000000000002, 3.23), ('compromising', ['E', 'C', 'A', 'B', 'D'], {'B': 10, 'C': 9, 'D': 9, 'A': 8, 'E': 4}, 0.62, 0.44, 3.1799999999999997, 3.23), ('compromising', ['B', 'C', 'A', 'E', 'D'], {'B': 13, 'C': 9, 'D': 9, 'A': 8, 'E': 1}, 0.62, 0.44, 3.1799999999999997, 3.23), ('compromising', ['D', 'C', 'A', 'E', 'B'], {'D': 13, 'B': 9, 'C': 9, 'A': 8, 'E': 1}, 0.48, 0.44, 3.27, 3.23), ('burying', ['C', 'A', 'E', 'B', 'D'], {'B': 10, 'C': 10, 'A': 9, 'D': 9, 'E': 2}, 0.72, 0.44, 3.15, 3.23), ('burying', ['C', 'A', 'E', 'B', 'D'], {'B': 1

[{'Voter': 1,
  'Strategic Options': [('compromising',
    ['A', 'C', 'E', 'B', 'D'],
    {'A': 10, 'B': 10, 'C': 9, 'D': 9, 'E': 2},
    0.69,
    0.44,
    3.0300000000000002,
    3.23),
   ('compromising',
    ['E', 'C', 'A', 'B', 'D'],
    {'B': 10, 'C': 9, 'D': 9, 'A': 8, 'E': 4},
    0.62,
    0.44,
    3.1799999999999997,
    3.23),
   ('compromising',
    ['B', 'C', 'A', 'E', 'D'],
    {'B': 13, 'C': 9, 'D': 9, 'A': 8, 'E': 1},
    0.62,
    0.44,
    3.1799999999999997,
    3.23),
   ('compromising',
    ['D', 'C', 'A', 'E', 'B'],
    {'D': 13, 'B': 9, 'C': 9, 'A': 8, 'E': 1},
    0.48,
    0.44,
    3.27,
    3.23),
   ('burying',
    ['C', 'A', 'E', 'B', 'D'],
    {'B': 10, 'C': 10, 'A': 9, 'D': 9, 'E': 2},
    0.72,
    0.44,
    3.15,
    3.23),
   ('burying',
    ['C', 'A', 'E', 'B', 'D'],
    {'B': 10, 'C': 10, 'A': 9, 'D': 9, 'E': 2},
    0.72,
    0.44,
    3.15,
    3.23),
   ('burying',
    ['C', 'A', 'E', 'B', 'D'],
    {'B': 10, 'C': 10, 'A': 9, 'D': 9, 'E': 2},
  